In [1]:

from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# Setup Ollama
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

parser = StrOutputParser()


In [2]:

prompt = PromptTemplate.from_template(
    "Translate to French: {text}"
)

parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({
    "text": "Good morning"
})

print(response)


Bonjour.


In [4]:

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define output structure
class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

# Ollama model
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Pydantic parser
parser = PydanticOutputParser(
    pydantic_object=ProductInfo
)

# Prompt
prompt = PromptTemplate(
    template="""
Extract the product name and price from the text.

Return only a JSON object containing:
- name: product name as a string
- price: price as a number in INR

Example:
{{
  "name": "Redmi Note 12",
  "price": 14999
}}

Text:
{text}

{format_instructions}
""",
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

# Force JSON output from Ollama
json_llm = llm.bind(format="json")

# Chain
chain = prompt | json_llm | parser

# Execute
response = chain.invoke({
    "text": "The Redmi Note 12 is available for ₹14,999."
})

print(response)
print("Product:", response.name)
print("Price:", response.price)


name='Redmi Note 12' price=14999.0
Product: Redmi Note 12
Price: 14999.0


In [5]:

from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser
)

prompt = PromptTemplate.from_template(
    "List 5 programming languages, comma-separated."
)

parser = CommaSeparatedListOutputParser()

chain = prompt | llm | parser

response = chain.invoke({})

print(response)


['Python', 'Java', 'JavaScript', 'C++', 'Ruby']


In [7]:

from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# Define schema
class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

# Ollama model
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Structured output
structured_llm = llm.with_structured_output(ProductInfo)

# Invoke model
response = structured_llm.invoke(
    "Extract product name and price from this text: "
    "The Redmi Note 12 is available for ₹14,999. "
    "Return the product information."
)

# Output
print(response)
print("Product Name:", response.name)
print("Price:", response.price)


name='Redmi Note 12' price=14999.0
Product Name: Redmi Note 12
Price: 14999.0


In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# Correct import for current LangChain versions
from langchain_classic.output_parsers.structured import (
    ResponseSchema,
    StructuredOutputParser
)

# Setup Ollama
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Define schema
schemas = [
    ResponseSchema(
        name="company",
        description="Name of the company"
    ),
    ResponseSchema(
        name="founder",
        description="Name of the founder"
    )
]

# Create parser
parser = StructuredOutputParser.from_response_schemas(
    schemas
)

# Create prompt
prompt = PromptTemplate(
    template="""
Extract the company and founder from the text.

Return only a JSON object.
Do not generate Python code.
Do not include explanations.

Text:
{text}

{format_instructions}
""",
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

# Ollama JSON mode
json_llm = llm.bind(format="json")

# Create chain
chain = prompt | json_llm | parser

# Execute
response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})

print(response)

{'company': 'Hope AI', 'founder': 'Ramisha Rani'}
